# SLURM scripts (Altamira)

In [1]:
import pandas as pd

In [3]:
df_files = pd.read_csv("project_selection.csv", index_col=0)
df_files.head()

,SubSet,FileName,sample,model_selection
1,J120,j1201_4.sm,True,True
7,J120,j1204_9.sm,True,True
23,J120,j12014_10.sm,True,True
30,J120,j12020_3.sm,True,True
37,J120,j12022_10.sm,True,True


In [ ]:
# Create SLURM scripts for running a set of jobs at Altamira cloud computing
# Random Forest, XGBoost, Support Vector Machine, Elastic Net, EBM

for model in ['rf','xgb','sv','en','ebm','fastai']:
    # time limit for each model
    if model in ['rf', 'xgb','sv','en']:
        time="5:00:00"
    elif model in ['fastai', 'ebm']:
        time="10:00:00"
    for file_name in df_files['FileName'].values:
        project_name = file_name.split('.')[0]
        with open(f"jobs/slurm_lhs_model_selection_{model}_project{project_name}.sh", "w") as f:
            f.write("#!/bin/bash\n")
            f.write(f"#SBATCH --job-name=lhs_msel_{model}_{project_name}"+"\n")
            f.write(f"#SBATCH --output=lhs_msel_{model}_{project_name}.out"+"\n")
            f.write(f"#SBATCH --error=lhs_msel_{model}_{project_name}.err"+"\n")
            f.write("#SBATCH --time="+time+"\n")
            f.write("#SBATCH --ntasks=1\n") # 1 tasks per job
            f.write("#SBATCH --cpus-per-task=16\n")
            f.write("#SBATCH --chdir=/home/uc27/uc27001/glassboxevm/\n")
            f.write('export OPENBLAS_NUM_THREADS=1\n')
            f.write('module purge\n')
            f.write('source $HOME/mlenv/bin/activate\n')
            f.write(f"srun -n 1 --ntasks-per-node=16 python3 ./model_selection/model_selection.py {model} {project_name}")  
        f.close()
